# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaAAND/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (client, query) pair from fact_content_query_90d, aggregated across however many distinct content items competed for that query in the 90-day window.

In [2]:
# This cell is for CODE (numbers, a query, a check).
#One row = one (client, query) pair from fact_content_query_90d, aggregated across however many distinct content items competed for that query in the 90-day window.
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")


rel = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
"""
con.sql(query).df()

result = con.sql(query).df()
print(result.shape)
result.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(0, 4)


,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


Feature (things a model would use as inputs):

content_count_per_query — number of distinct content_hash_ids sharing a query_hash_id (this is your core cannibalization signal, computed via grouping, not a raw column)
impressions_90d, clicks_90d — per (client, content, query) volume
avg_position_90d — ranking position, useful to see if competing pages cluster at similar/different ranks
rare_impressions_share, anonymized_impressions_share — data-quality-adjacent, but also informative as features (high anonymized share = noisier signal)

Label / proxy (what you're predicting or ranking by):

cannibalization_severity — a derived score combining content_count_per_query and how evenly impressions_90d/clicks_90d are split across the competing content items for that query

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify all fields mentioned in the contract actually exist in the tables
cols_query90d = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_query_90d.parquet') LIMIT 1").df()['column_name'].tolist()

used_fields = ['client_hash_id','content_hash_id','query_hash_id','impressions_90d',
               'clicks_90d','avg_position_90d','rare_impressions_share',
               'anonymized_impressions_share','query_char_count','query_token_count']

missing = [f for f in used_fields if f not in cols_query90d]
print("Missing fields:", missing if missing else "None — all fields confirmed")


Missing fields: None — all fields confirmed


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1 — grain check (expect empty)
q1 = f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY 1,2,3 HAVING COUNT(*) > 1
"""
print(con.sql(q1).df())

# Query 2 — row count + date span
q2 = f"""
SELECT COUNT(*) total_rows, MIN(report_date) earliest, MAX(report_date) latest
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
"""
print(con.sql(q2).df())

# Query 3 — availability with IS TRUE
q3 = f"""
SELECT COUNT(*) total_rows,
       SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) gsc_available
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
"""
print(con.sql(q3).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
   total_rows   earliest     latest
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available
0     9841378      3611061.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q4 = f"""
SELECT
    gsc_data_start,
    ga4_data_start,
    COUNT(*) AS num_clients
FROM read_parquet('{rel}/dim_clients.parquet')
GROUP BY gsc_data_start, ga4_data_start
ORDER BY gsc_data_start
"""
print(con.sql(q4).df())

   gsc_data_start ga4_data_start  num_clients
0      2025-01-27     2025-10-29            2
1      2025-02-11     2026-03-24            1
2      2025-03-11     2026-03-06            1
3      2025-06-07            NaT            1
4      2025-06-18     2025-11-15            1
5      2025-06-21     2026-02-19            1
6      2025-06-21     2026-02-20            1
7      2025-06-29     2025-11-09            1
8      2025-07-01     2026-02-19            1
9      2025-07-06     2026-02-19            1
10     2025-07-07     2026-02-19            1
11     2025-07-17     2026-02-19            1
12     2025-07-21     2026-02-19            1
13     2025-07-28            NaT            1
14     2025-07-29            NaT            1
15     2025-09-24            NaT            2
16     2025-09-24     2026-02-19            4
17     2025-09-24     2025-10-29            1
18     2025-09-27            NaT            1
19     2025-10-11     2026-03-11            1
20     2025-10-13     2025-11-09  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.